In [16]:
import os
import json
import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
import anthropic
import re

load_dotenv()  # reads .env in the current working directory

# ----------------------------------------------------------------------
# 1. Client & model
# ----------------------------------------------------------------------

api_key = os.getenv("anthropic_api_key")
client = anthropic.Anthropic(api_key=api_key)

MODEL_ID = "claude-haiku-4-5-20251001"  # dated, pinned ID -> reproducible results

`claude-haiku-4-5` is an **alias** (no date mentioned) — a convenience pointer that always resolves to whichever dated snapshot is currently the **latest** one under the "Haiku 4.5" minor version.
- It isn't a model itself — it's a **pointer** that Anthropic re-aims, not something you control
- Suppose today, it resolves to the **same model** as the pinned ID, since `-20251001` happens to be the latest snapshot right now
- If Anthropic ships a new Haiku 4.5 snapshot next month, the alias **silently starts using the new one** — same code on your end, different model underneath
- Good for **quick experiments** — always on the newest patch with zero code changes

`claude-haiku-4-5-20251001` is the actual **pinned snapshot**.
- Its weights and behavior are **fixed forever** under that exact ID — Anthropic never silently updates an existing dated ID
- If they improve Haiku 4.5, it ships as a **new** dated snapshot (and the alias then quietly starts pointing to that new one instead)
- Suppose today, it's the **same model** as the alias
- If Anthropic ships a new Haiku 4.5 snapshot next month, this ID still points to the **exact Oct 2025 snapshot** — nothing changes underneath you
- Good for **reproducible benchmarks** and production, where any change in output needs to come from your prompt — never from a silent model swap

A **system prompt** is a separate instruction channel in the Messages API, passed through the dedicated `system` parameter — not inside the `messages` array — that sets the model's role and rules before it sees any actual input.
- It sits **outside** the `messages` array, so it's never treated as a turn the model is replying to
- It persists for the **entire call** — you set it once per request, not once per email
- It carries more **authority** than a user message — the model treats it as instructions from you, the developer, rather than as data to act on
- Suppose today, `SYSTEM_PROMPT` stays **exactly the same** across every single email you classify
- If the email content changes, only `user_message` changes — `SYSTEM_PROMPT` never does
- Good for domain knowledge, label definitions, and output-format rules — anything that should hold true no matter which email comes in

In [22]:
SYSTEM_PROMPT = """You classify customer emails for Bajaj Finance into one of three categories:
  - FD: the email is about a Fixed Deposit account
  - Non-FD: the email is about anything else
  - Multiple Category: the email is about both FD and non-FD topics

Respond with ONLY this JSON format, nothing else: 
{"label": "FD" | "Non-FD" | "Multiple Category", "reason": "<one short sentence>"}
"""

In [23]:
def _strip_code_fences(text: str) -> str:
    """Models often wrap JSON in ```json ... ``` even when told not to.
    Strip that wrapping if present, so json.loads() still works."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()

Send one email to Haiku 4.5 and return the parsed classification

In [37]:
def classify_email(subject: str, content: str) -> dict:
    
    user_message = f"Subject: {subject}\n\nBody: {content}"
 
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=150,              # classification output is short -> cap it deliberately
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": user_message}
        ],
    )
 
    raw_text = response.content[0].text.strip()
    cleaned_text = _strip_code_fences(raw_text)
    print("\n" + "=" * 60)
    print("HAIKU 4.5 Raw OUTPUT Received")
    print("=" * 60)
    print(response)
    try:
        return json.loads(cleaned_text)
    except json.JSONDecodeError:
        # Model didn't return clean JSON -> surface the raw text instead of crashing.
        return {"label": "PARSE_ERROR", "reason": raw_text}

In [38]:
if __name__ == "__main__":
    # Load the real dataset and pick ONE email to test the call end-to-end.
    # Update the path below if your CSV lives elsewhere.
    df = pd.read_csv("../data/fd_dataset_messy.csv")
    sample = df.iloc[5]   # a real FD email containing reference number BJ2019FD7717
 
    print("=" * 60)
    print("INPUT EMAIL")
    print("=" * 60)
    print(f"\nSubject : {sample['subject']}")
    print(f"\nContent : {sample['content']}")
    print(f"\nTrue label (from dataset) : {sample['label']}")
 
    result = classify_email(sample["subject"], sample["content"])
 
    print("\n" + "=" * 60)
    print("HAIKU 4.5 Final OUTPUT")
    print("=" * 60)
    print(json.dumps(result, indent=2))

INPUT EMAIL

Subject : Help

Content : Hello ji, Yeh second baar likh raha hoon. My funds with your company is stuck. The period was over in December but nothing came to my bank. Please check BJ2019FD7717. Jaldi kuch karo please. Thank you. Shobha Chopra | 9686667744

True label (from dataset) : FD

HAIKU 4.5 Raw OUTPUT Received
Message(id='msg_01Mh7Eh8ryuBSGgQBLN7gvt1', container=None, content=[TextBlock(citations=None, text='```json\n{"label": "FD", "reason": "Customer is inquiring about a matured Fixed Deposit account (reference number BJ2019FD7717) where funds have not been credited to their bank account."}\n```', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=192, output_tokens=55, ou

The **raw output** is the full `Message` object that `client.messages.create()` returns — `response.content[0].text` in your script only pulls out one small piece of it.
- The actual text lives inside **`content`**, which is always a list of blocks — `content[0]` is the first block, `.text` is the string inside it
- **`model`** confirms the exact pinned snapshot that actually served the request — your receipt for what really ran, especially useful if you ever call through an alias instead
- **`stop_reason`** tells you **why** generation stopped — `"end_turn"` means it finished on its own; `"max_tokens"` would mean the response got cut off mid-JSON
- `output_tokens=56` stayed well under your `max_tokens=150` cap, which is exactly why `stop_reason` read `"end_turn"` instead of `"max_tokens"`
- **`usage`** is the billing record for that one call — `input_tokens` + `output_tokens` are what you're actually charged for, at Haiku 4.5 rates roughly **$0.0005** for this single email
- Good for verifying cost, catching truncated responses, and confirming which model snapshot answered — none of which `response.content[0].text` alone can tell you